# Triagegeist: Clinical Safety-Aware ED Triage Prediction

**Author**: Thiebaut Schirmer
**Date**: 2026-04-16
**Model**: Fine-tuned Bio_ClinicalBERT + gradient-boosted blend + Nelder-Mead thresholds
**Kaggle**: Laitinen-Fredriksson Foundation ED Triage Acuity Competition

---

**One-line summary**: An ED triage acuity classifier validated on 80 k synthetic (Kaggle)
and 418 k real clinical (MIMIC-IV-ED) encounters, with explicit clinical-safety
optimization (asymmetric under-triage cost), intersectional bias auditing,
and SHAP explainability. Matches live-patient nurse-to-expert inter-rater
reliability on real data (κ = 0.701 vs published 0.694).

> **⚠️ Reproduction note**: This notebook requires Kaggle's T4 GPU (Settings → Accelerator)
> and **internet access enabled** (to download Bio_ClinicalBERT from HuggingFace).
> Also attach the supplementary dataset `triagegeist-mimic-supplementary` (Settings → Add data).
> Runtime: ~35–45 minutes end-to-end.

In [ ]:
# Kaggle-specific install (only packages not pre-installed)
%pip install -q shap --quiet

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import json, warnings, time
from pathlib import Path
from IPython.display import Image, display, Markdown

warnings.filterwarnings('ignore')

# ── Paths ────────────────────────────────────────────────────────────────
def find_kaggle_input(folder_name):
    """Locate a Kaggle input folder regardless of mount layout (flat vs nested)."""
    matches = list(Path('/kaggle/input').rglob(folder_name))
    return matches[0] if matches else Path(f'/kaggle/input/{folder_name}')

COMP_DATA = find_kaggle_input('triagegeist')
SUPP_DATA = find_kaggle_input('triagegeist-mimic-supplementary')
OUTPUT    = Path('/kaggle/working')

print(f'Competition data:  {COMP_DATA}  exists={COMP_DATA.exists()}')
print(f'Supplementary data: {SUPP_DATA}  exists={SUPP_DATA.exists()}')

# ── Plotting style ───────────────────────────────────────────────────────
plt.rcParams.update({
    'figure.figsize': (10, 6), 'font.size': 11,
    'axes.spines.top': False, 'axes.spines.right': False,
    'axes.titlesize': 13,
})

# ── Van Gogh Almond Blossoms palette ─────────────────────────────────────
# Canonical source: notebook/vangogh_palette.py in the repo. Pasted inline
# because Kaggle notebooks cannot import sibling Python files.
VANGOGH = {
    'blue':  '#3D6B7E',   # deep navy/teal (matches LaTeX vg-heading)
    'gold':  '#C4A882',   # warm golden ochre (vg-blossom)
    'sage':  '#6B8E6B',   # muted sage green (vg-leaf)
    'rust':  '#E8525A',   # warm rust accent (severe / under-triage)
    'cream': '#F2EBD9',   # warm cream figure background
    'text':  '#3A3530',   # dark warm body text
}

ESI_COLORS = {
    1: VANGOGH['rust'],   # resuscitation
    2: VANGOGH['gold'],   # emergent
    3: VANGOGH['sage'],   # urgent
    4: VANGOGH['blue'],   # less urgent
    5: VANGOGH['text'],   # non-urgent
}

# 1. Executive summary

This notebook presents a **clinical decision-support system** for 5-level Emergency
Severity Index (ESI) triage, developed and validated across two datasets:

1. **Kaggle competition data**: 80 k synthetic ED visits
2. **MIMIC-IV-ED** (Xie et al. 2022): 418 100 real ED visits from Beth Israel Deaconess

## Headline findings

**Synthetic data has label leakage.** Severity keywords in `chief_complaint_raw`
(`mild`, `severe`, `massive`, `moderate`) partition near-disjointly across ESI
levels. Any NLP model achieves ~0.998 F1. Honest tabular modelling caps at ~0.873.
**The leaderboard metric measures keyword recovery, not clinical prediction.**

**On real clinical data, my pipeline achieves**:

| Metric | Value | Benchmark |
|---|---|---|
| Macro F1 (5-fold CV) | **0.597** | 5-class ESI at 418 k real ED visits |
| Cohen's quadratic κ | **0.701** | = live-patient nurse-to-expert κ (Mirhaghi 2015: 0.694) |
| AUROC (macro-OVR) | **0.911** | Raita et al. 2019 binary AUC 0.89 |
| N training encounters | 418 100 | Sax et al. 2023: 5.3 M binary triage study |

**Three findings worth flagging:**

1. Training-time asymmetric class weights and post-hoc threshold multipliers
   occupy nearly the same F1-vs-UT Pareto frontier. Layering them undoes the
   safety benefit of either alone.
2. The ACS-COT 5% target is reachable but costly: a per-class threshold
   multiplier of 3.0× achieves UT = 4.87% at F1 = 0.459, roughly a 23%
   relative F1 reduction from the unconstrained optimum.
3. Compensated shock is empirically visible in the cohort: 49.5% of true
   ESI-1/2 patients aged 18–30 present with heart rate and systolic BP both
   in normal range, vs 37.2% of patients aged 66+. This is the most plausible
   mechanism behind the age-based disparity in under-triage rates.

## Companion report

A 125-page LaTeX report covering the full methodology, ablations, and chapter-level discussion is included in the repository at `report/triagegeist_report.pdf` ([GitHub](https://github.com/Thiebauts/kaggle-triagegeist/blob/main/report/triagegeist_report.pdf)). This notebook is the executable short-form summary; the report is the long-form discussion.

## Clinical context: ESI and RETTS

The Emergency Severity Index (ESI) is the dominant triage system in North American EDs. The Rapid Emergency Triage and Treatment System (RETTS; Widgren & Jourak, *J Emerg Med* 2011;40:623–628) is the dominant system in Scandinavian EDs and uses a two-step algorithm where vital-sign priority and chief-complaint priority are combined via a *max* function: abnormal vitals always override a benign-sounding complaint, structurally encoding under-triage protection.

Two connections to this work:

1. **Algorithmic safety design.** RETTS's *max* function is loosely analogous to the asymmetric training weights explored here: both encode the institutional judgement that the cost of missing a critical patient outweighs the cost of over-allocating resources. The analogy is structural, not exact.
2. **Shared age-adjustment limitation.** Wireklint et al. (*Scand J Trauma Resusc Emerg Med* 2022;30:12) found that RETTS's discriminative ability decreases with age and that age is an independent mortality risk factor across all RETTS levels. ESI has the same blind spot. My age-stratified vital-sign flags and age-adjusted z-scores address this directly; see §4 and §8.

## How this notebook maps to the scoring rubric

- **Clinical Relevance (25)**: §2 motivation; §7 ACS-COT compliance; §8 EU AI Act.
- **Technical Quality (30)**: §3 ClinicalBERT fine-tune; §4 feature engineering; §5 blend + threshold opt.
- **Documentation (20)**: full methodology in cells below; supplementary dataset for reproducibility.
- **Insight & Findings (15)**: three findings flagged above.
- **Novelty & Impact (10)**: substitute-not-complement and compensated-shock findings.

# 2. The synthetic data problem

Before modelling, I characterised the Kaggle dataset and discovered that the
free-text `chief_complaint_raw` column encodes the label. This section proves it
in three ways: severity-keyword distribution, chief-complaint-system chi-squared,
and exact-match analysis between train and test.

In [ ]:
# Load and join Kaggle data — pain_score=-1 is a sentinel for 'not recorded'
train = pd.read_csv(COMP_DATA / 'train.csv')
test  = pd.read_csv(COMP_DATA / 'test.csv')
complaints = pd.read_csv(COMP_DATA / 'chief_complaints.csv',
                          usecols=['patient_id', 'chief_complaint_raw'])
history    = pd.read_csv(COMP_DATA / 'patient_history.csv')

for df in (train, test):
    if 'pain_score' in df.columns:
        df['pain_score'] = df['pain_score'].replace(-1, np.nan)

# Drop post-triage columns from train to prevent target leakage
for col in ('disposition', 'ed_los_hours'):
    if col in train.columns:
        train = train.drop(columns=col)

df_train = train.merge(history, on='patient_id', how='left') \
                .merge(complaints, on='patient_id', how='left')
df_test = test.merge(history, on='patient_id', how='left') \
              .merge(complaints, on='patient_id', how='left')

print(f'Train: {df_train.shape}  |  Test: {df_test.shape}')
print(f'History cols: {len([c for c in history.columns if c.startswith("hx_")])}')
print(f'\nClass distribution:')
print(df_train['triage_acuity'].value_counts(normalize=True).sort_index().round(3))

## 2.1 Severity keyword distribution: evidence of synthetic label encoding

The Kaggle synthetic data over-samples the middle of the ESI scale and
almost omits the rare classes. Plotting it inline (no external image)
keeps the source of truth in the notebook.

In [ ]:
counts = df_train['triage_acuity'].value_counts(normalize=True).sort_index()
fig, ax = plt.subplots(figsize=(7.5, 3.8))
bars = ax.bar([f'ESI-{i}' for i in counts.index], counts.values * 100,
              color=[ESI_COLORS[i] for i in counts.index], edgecolor='white',
              linewidth=1.2)
for bar, val in zip(bars, counts.values * 100):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.8,
            f'{val:.1f}%', ha='center', fontsize=10, color='#3A3530')
ax.set_ylabel('Proportion of visits (%)')
ax.set_title('Kaggle synthetic ESI distribution', fontsize=12,
             color='#3A3530', pad=12)
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
ax.set_facecolor('#F2EBD9'); fig.patch.set_facecolor('#F2EBD9')
plt.tight_layout(); plt.show()

In [ ]:
KEYWORDS = ['mild', 'moderate', 'severe', 'acute', 'critical',
            'stable', 'massive', 'extreme', 'slight', 'minor',
            'significant', 'life-threatening']

kw_table = pd.DataFrame()
for kw in KEYWORDS:
    for esi in range(1, 6):
        mask = df_train['triage_acuity'] == esi
        pct = (df_train.loc[mask, 'chief_complaint_raw']
                       .fillna('').str.contains(kw, case=False, na=False)
                       .mean() * 100)
        kw_table.loc[kw, f'ESI-{esi}'] = round(pct, 1)

print('Percentage of rows per ESI level containing each keyword:')
kw_table

Note how `mild` appears exclusively in ESI-4/5 (lower acuity), `severe` and
`massive` concentrate in ESI-1/2 (higher acuity), and `moderate` peaks in ESI-3.
**Real chief complaints do not partition this cleanly across an ordinal acuity
scale.** This is diagnostic of synthetic label encoding.

Figure from my detailed analysis:

In [ ]:
import matplotlib.colors as mcolors
vg_cmap = mcolors.LinearSegmentedColormap.from_list(
    'vg_rust', ['#F2EBD9', VANGOGH['rust']])  # cream to rust

fig, ax = plt.subplots(figsize=(7.5, 5.5))
im = ax.imshow(kw_table.values, cmap=vg_cmap, aspect='auto')
ax.set_xticks(range(5), [f'ESI-{i}' for i in range(1, 6)])
ax.set_yticks(range(len(kw_table)), kw_table.index)
for i in range(kw_table.shape[0]):
    for j in range(kw_table.shape[1]):
        v = kw_table.values[i, j]
        c = 'white' if v > kw_table.values.max() * 0.5 else '#3A3530'
        ax.text(j, i, f'{v:.1f}', ha='center', va='center', fontsize=9, color=c)
ax.set_title('Severity keywords by ESI level: synthetic label fingerprint',
             fontsize=11, color='#3A3530', pad=12)
plt.colorbar(im, ax=ax, label='% of ESI-level rows')
ax.set_facecolor('#F2EBD9'); fig.patch.set_facecolor('#F2EBD9')
plt.tight_layout(); plt.show()

## 2.2 `chief_complaint_system` has zero predictive power

In [ ]:
from scipy.stats import chi2_contingency
ct = pd.crosstab(df_train['chief_complaint_system'], df_train['triage_acuity'])
chi2, p, dof, _ = chi2_contingency(ct)
print(f'chi-squared = {chi2:.1f}, p-value = {p:.3e}, dof = {dof}')
print(f'\nAcuity distribution within each system (each row sums to 1.0):')
print((ct.div(ct.sum(axis=1), axis=0) * 100).round(1).iloc[:6])

Despite a nominally significant p-value from the sheer sample size, the
acuity distribution *within* each body-system category is near-uniform,
the categorical column captures no clinical signal.

## 2.3 Train/test exact-match analysis

In [ ]:
train_unique = set(df_train['chief_complaint_raw'].dropna().unique())
test_unique  = set(df_test['chief_complaint_raw'].dropna().unique())
overlap = train_unique & test_unique
print(f'Unique complaints in train: {len(train_unique):,}')
print(f'Unique complaints in test:  {len(test_unique):,}')
print(f'Overlap: {len(overlap):,} ({100*len(overlap)/len(test_unique):.1f}% of test)')

# Majority-label consistency for overlapping strings
maj = (df_train.dropna(subset=['chief_complaint_raw'])
               .groupby('chief_complaint_raw')['triage_acuity']
               .agg(lambda s: s.mode().iloc[0]))
matched = df_train[df_train['chief_complaint_raw'].isin(overlap)].copy()
matched['maj'] = matched['chief_complaint_raw'].map(maj)
print(f"\nMajority-label accuracy on matched train rows: "
      f"{(matched['maj'] == matched['triage_acuity']).mean():.4f}")

In [ ]:
verbatim_pct = 100 * len(overlap) / len(test_unique)
novel_pct = 100 - verbatim_pct
fig, ax = plt.subplots(figsize=(8, 1.5))
ax.barh([0], [verbatim_pct], color=VANGOGH['rust'],
        label=f'Verbatim match ({verbatim_pct:.1f}%)')
ax.barh([0], [novel_pct], left=[verbatim_pct], color=VANGOGH['cream'],
        edgecolor=VANGOGH['rust'], linewidth=1,
        label=f'Novel ({novel_pct:.1f}%)')
ax.set_xlim(0, 100); ax.set_yticks([])
ax.set_xlabel('% of unique test complaints')
ax.legend(loc='lower center', bbox_to_anchor=(0.5, -1.0), ncol=2, frameon=False)
ax.set_title('Test complaints already seen verbatim in training data',
             fontsize=11, color='#3A3530', pad=10)
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
ax.set_facecolor('#F2EBD9'); fig.patch.set_facecolor('#F2EBD9')
plt.tight_layout(); plt.show()

**Conclusion**: the Kaggle leaderboard measures keyword recovery. I
therefore validate my methodology on real clinical data (MIMIC-IV-ED, §6),
while submitting two CSVs to the competition: the leakage-recovering full
pipeline (leaderboard-valid) and an honest tabular model (what a clinical
system would actually deploy).

# 3. Fine-tuning Bio_ClinicalBERT on chief complaints

I fine-tune [`emilyalsentzer/Bio_ClinicalBERT`](https://huggingface.co/emilyalsentzer/Bio_ClinicalBERT)
(Alsentzer et al. 2019), a transformer pre-trained on MIMIC-III clinical notes,
on the 80 000 Kaggle training chief complaints. On MIMIC-IV-ED, fine-tuned
ClinicalBERT improved F1 by +0.015 over frozen MiniLM (validated across 9
candidate embedding models); I reproduce the pipeline here with adapted
hyperparameters for the shorter Kaggle complaints.

**Kaggle config**: 3 epochs, AdamW lr=2e-5, batch=64, max_len=64, mean-pool
+ 5-class head with 0.1 dropout. ~15–18 min on Kaggle T4.

**MIMIC config** (as reported): batch=16, max_len=128, with FP16 mixed precision,
gradient clipping, and best-model selection by validation F1. The Kaggle
complaints are shorter and simpler than real clinical text, so the reduced
sequence length is appropriate here.

In [ ]:
%%time
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if not torch.cuda.is_available():
    print('[WARN] GPU not detected — fine-tuning will be slow')

MODEL_NAME = 'emilyalsentzer/Bio_ClinicalBERT'
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


class TriageDataset(Dataset):
    def __init__(self, texts, labels=None, max_len=64):
        self.texts = texts
        self.labels = labels
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = tokenizer(self.texts[idx], truncation=True, padding='max_length',
                        max_length=self.max_len, return_tensors='pt')
        item = {k: v.squeeze(0) for k, v in enc.items()}
        if self.labels is not None:
            item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item


class ClinicalBERTClassifier(nn.Module):
    def __init__(self, model_name, n_classes=5):
        super().__init__()
        self.bert = AutoModel.from_pretrained(model_name)
        self.classifier = nn.Linear(self.bert.config.hidden_size, n_classes)
        self.dropout = nn.Dropout(0.1)

    def forward(self, input_ids, attention_mask, **kwargs):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        # Mean pooling
        mask = attention_mask.unsqueeze(-1).float()
        pooled = (outputs.last_hidden_state * mask).sum(1) / mask.sum(1)
        return self.classifier(self.dropout(pooled)), pooled


# Union of train + test complaints, keep order aligned with df_train/df_test
all_rows = pd.concat([
    df_train[['patient_id', 'chief_complaint_raw', 'triage_acuity']],
    df_test[['patient_id', 'chief_complaint_raw']].assign(triage_acuity=-1),
], ignore_index=True)
all_rows['chief_complaint_raw'] = all_rows['chief_complaint_raw'].fillna('').astype(str)

train_mask = all_rows['triage_acuity'] > 0
tr_texts = all_rows.loc[train_mask, 'chief_complaint_raw'].tolist()
tr_labels = (all_rows.loc[train_mask, 'triage_acuity'] - 1).tolist()  # 0-indexed

train_ds = TriageDataset(tr_texts, tr_labels)
train_loader = DataLoader(train_ds, batch_size=64, shuffle=True, num_workers=2)

model = ClinicalBERTClassifier(MODEL_NAME, n_classes=5).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5, weight_decay=0.01)
criterion = nn.CrossEntropyLoss()

EPOCHS = 3
for epoch in range(EPOCHS):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for batch in train_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        logits, _ = model(input_ids, attention_mask)
        loss = criterion(logits, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * len(labels)
        correct += (logits.argmax(1) == labels).sum().item()
        total += len(labels)

    print(f'Epoch {epoch+1}/{EPOCHS}  loss={total_loss/total:.4f}  '
          f'train_acc={correct/total:.4f}')

In [ ]:
%%time
# Extract embeddings for ALL complaints (train + test), keep row order
model.eval()
all_ds = TriageDataset(all_rows['chief_complaint_raw'].tolist())
all_loader = DataLoader(all_ds, batch_size=128, shuffle=False, num_workers=2)

all_emb = []
with torch.no_grad():
    for batch in all_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        _, pooled = model(input_ids, attention_mask)
        all_emb.append(pooled.cpu().numpy())

all_emb = np.vstack(all_emb)
print(f'Embeddings shape: {all_emb.shape}  dtype={all_emb.dtype}')

In [ ]:
# PCA to 50 dimensions
# NOTE: In the MIMIC pipeline, PCA is fit *within each CV fold* to prevent
# leakage. Here we fit globally for simplicity — on the Kaggle synthetic data
# the leakage from global PCA is negligible because the text already encodes
# the label. For a clinical deployment, per-fold PCA is required.
from sklearn.decomposition import PCA

pca = PCA(n_components=50, random_state=42)
emb_pca = pca.fit_transform(all_emb)
print(f'Explained variance (cumulative): {pca.explained_variance_ratio_.sum():.3f}')

# Split back into train and test, keeping row alignment
train_emb = emb_pca[:len(df_train)]
test_emb  = emb_pca[len(df_train):]
assert len(train_emb) == len(df_train)
assert len(test_emb) == len(df_test)
print(f'train_emb: {train_emb.shape}  test_emb: {test_emb.shape}')

# Free GPU memory
del model, all_emb, emb_pca
torch.cuda.empty_cache() if torch.cuda.is_available() else None

# 4. Feature engineering

I build ~55 tabular features (vitals, engineered composites such as shock index,
MAP, NEWS2-partial, pulse pressure), missingness flags, cyclic arrival time,
demographics, and 25 `hx_*` comorbidity flags), then concatenate with the 50
ClinicalBERT PCA dimensions. **No post-triage columns** (`disposition`,
`ed_los_hours`) are used. The exact tabular feature count printed below
depends on which columns the Kaggle data provides; the MIMIC pipeline (§6)
uses a larger 65-feature set with additional medication-class flags and
Charlson comorbidity weights.

In [ ]:
VITAL_RANGES = {
    'heart_rate': (60, 100), 'systolic_bp': (90, 140), 'diastolic_bp': (60, 90),
    'temperature_c': (36.1, 38.0), 'respiratory_rate': (12, 20), 'spo2': (95, 100),
}

def n2_rr(v):
    if pd.isna(v): return np.nan
    if v <= 8: return 3
    if v <= 11: return 1
    if v <= 20: return 0
    if v <= 24: return 2
    return 3

def n2_o2(v):
    if pd.isna(v): return np.nan
    if v <= 91: return 3
    if v <= 93: return 2
    if v <= 95: return 1
    return 0

def n2_sbp(v):
    if pd.isna(v): return np.nan
    if v <= 90: return 3
    if v <= 100: return 2
    if v <= 110: return 1
    if v <= 219: return 0
    return 3

def n2_hr(v):
    if pd.isna(v): return np.nan
    if v <= 40: return 3
    if v <= 50: return 1
    if v <= 90: return 0
    if v <= 110: return 1
    if v <= 130: return 2
    return 3

def n2_temp(v):
    if pd.isna(v): return np.nan
    if v <= 35.0: return 3
    if v <= 36.0: return 1
    if v <= 38.0: return 0
    if v <= 39.0: return 1
    return 2


def engineer_features(data):
    f = pd.DataFrame(index=data.index)

    # Raw + pre-engineered vitals present in train.csv
    for c in ['systolic_bp', 'diastolic_bp', 'mean_arterial_pressure',
              'pulse_pressure', 'heart_rate', 'respiratory_rate',
              'temperature_c', 'spo2', 'gcs_total', 'pain_score',
              'shock_index', 'news2_score', 'bmi', 'weight_kg', 'height_cm',
              'age', 'num_prior_ed_visits_12m', 'num_prior_admissions_12m',
              'num_active_medications', 'num_comorbidities']:
        if c in data.columns:
            f[c] = pd.to_numeric(data[c], errors='coerce')

    # Engineered vitals (fill in if the dataset didn't already compute)
    if 'shock_index' not in f.columns or f['shock_index'].isna().all():
        f['shock_index'] = f['heart_rate'] / f['systolic_bp'].replace(0, np.nan)
    if 'mean_arterial_pressure' not in f.columns:
        f['mean_arterial_pressure'] = f['diastolic_bp'] + (f['systolic_bp'] - f['diastolic_bp']) / 3.0
    if 'pulse_pressure' not in f.columns:
        f['pulse_pressure'] = f['systolic_bp'] - f['diastolic_bp']

    # We keep BOTH news2_score (pre-computed in the CSV, may use different thresholds)
    # and news2_recomputed (our own NEWS2 2017 implementation) as features. The tree
    # models can choose whichever discriminates better; if they agree, no harm.
    f['news2_recomputed'] = (
        f['respiratory_rate'].apply(n2_rr).fillna(0)
        + f['spo2'].apply(n2_o2).fillna(0)
        + f['systolic_bp'].apply(n2_sbp).fillna(0)
        + f['heart_rate'].apply(n2_hr).fillna(0)
        + f['temperature_c'].apply(n2_temp).fillna(0)
    )

    # Missingness flags (clinically meaningful — missing vitals in ESI-1 often
    # indicate the patient was too unstable to measure)
    for c in ['heart_rate', 'systolic_bp', 'diastolic_bp',
              'temperature_c', 'respiratory_rate', 'spo2']:
        if c in data.columns:
            f[f'{c}_missing'] = data[c].isna().astype(int)

    # Count of abnormal vitals
    ab = np.zeros(len(data), dtype=int)
    for c, (lo, hi) in VITAL_RANGES.items():
        if c in data.columns:
            v = pd.to_numeric(data[c], errors='coerce')
            ab += ((v < lo) | (v > hi)).fillna(0).astype(int).values
    f['n_abnormal_vitals'] = ab

    # Cyclic arrival hour
    if 'arrival_hour' in data.columns:
        h = pd.to_numeric(data['arrival_hour'], errors='coerce').fillna(12)
        f['arrival_hour_sin'] = np.sin(2 * np.pi * h / 24.0)
        f['arrival_hour_cos'] = np.cos(2 * np.pi * h / 24.0)

    # 25 hx_* binary flags
    for c in [col for col in data.columns if col.startswith('hx_')]:
        f[c] = pd.to_numeric(data[c], errors='coerce').fillna(0).astype(int)

    # Label-encoded categoricals
    for c in ['sex', 'language', 'insurance_type', 'arrival_mode',
              'arrival_day', 'arrival_season', 'shift', 'transport_origin',
              'chief_complaint_system', 'mental_status_triage', 'pain_location',
              'age_group']:
        if c in data.columns:
            f[c] = data[c].astype('category').cat.codes  # -1 for NaN

    return f


# Build feature matrices for train and test
Xtr_tab = engineer_features(df_train)
Xte_tab = engineer_features(df_test)

# Align columns (in case either side is missing a column)
for c in Xtr_tab.columns:
    if c not in Xte_tab.columns:
        Xte_tab[c] = 0
Xte_tab = Xte_tab[Xtr_tab.columns]

print(f'Tabular features: {Xtr_tab.shape[1]}')

# Concatenate with ClinicalBERT PCA embeddings
Xtr = np.hstack([Xtr_tab.values.astype(np.float32), train_emb.astype(np.float32)])
Xte = np.hstack([Xte_tab.values.astype(np.float32), test_emb.astype(np.float32)])
y = df_train['triage_acuity'].values
FEATURE_COLS = list(Xtr_tab.columns) + [f'emb_pca_{i}' for i in range(50)]

print(f'Full feature matrix: Xtr={Xtr.shape}  Xte={Xte.shape}')
print(f'Total feature count: {len(FEATURE_COLS)}')

# Sanity check: sum(hx_*) should match num_comorbidities
hx_cols = [c for c in Xtr_tab.columns if c.startswith('hx_')]
if hx_cols and 'num_comorbidities' in Xtr_tab.columns:
    agree = (Xtr_tab[hx_cols].sum(axis=1) == Xtr_tab['num_comorbidities']).mean()
    print(f'Sanity: sum(hx_*) == num_comorbidities in {agree*100:.1f}% of rows')

# 5. Model training: stacked blend with 5-fold CV

I train a blend of 7 heterogeneous models on the **Kaggle synthetic data**
(2 × LightGBM, 2 × XGBoost, CatBoost, LogisticRegression, RandomForest),
stack via a LogisticRegression meta-learner on OOF probabilities, then apply
Nelder-Mead threshold optimisation to maximise macro F1.

**Note**: the MIMIC-IV-ED pipeline (§6) uses a different, streamlined blend
of 5 models (4 LightGBM variants + 1 internally stacked LGB+XGB+CB ensemble)
with convex weight optimisation. The Kaggle blend here includes additional
model families (LR, RF) because the synthetic data's simpler structure allows
them to contribute meaningfully.

**Expected runtime on Kaggle**: ~12 min for the full blend.

In [ ]:
%%time
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, cohen_kappa_score, confusion_matrix
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from scipy.optimize import minimize
import lightgbm as lgb
import xgboost as xgb

try:
    from catboost import CatBoostClassifier
    HAS_CB = True
except ImportError:
    HAS_CB = False
    print('[info] CatBoost not available — blend will use 6 models')

# Impute NaNs once (for LR/RF which don't handle NaN)
imp = SimpleImputer(strategy='median')
Xtr_i = imp.fit_transform(Xtr)
Xte_i = imp.transform(Xte)

CLASSES = np.array([1, 2, 3, 4, 5])
K = len(CLASSES)
N = len(y)
MIN_CLS = int(CLASSES.min())

model_defs = [
    ('lgbm1', lambda: lgb.LGBMClassifier(
        n_estimators=200, learning_rate=0.05, num_leaves=63,
        subsample=0.8, colsample_bytree=0.8, bagging_freq=1,
        n_jobs=-1, random_state=42, verbose=-1)),
    ('lgbm2', lambda: lgb.LGBMClassifier(
        n_estimators=300, learning_rate=0.03, num_leaves=127,
        subsample=0.8, colsample_bytree=0.8, bagging_freq=1,
        n_jobs=-1, random_state=42, verbose=-1)),
    ('xgb1', lambda: xgb.XGBClassifier(
        n_estimators=200, learning_rate=0.05, max_depth=6,
        subsample=0.8, colsample_bytree=0.8,
        objective='multi:softprob', eval_metric='mlogloss',
        n_jobs=-1, random_state=42, verbosity=0)),
    ('xgb2', lambda: xgb.XGBClassifier(
        n_estimators=300, learning_rate=0.03, max_depth=8,
        subsample=0.8, colsample_bytree=0.8,
        objective='multi:softprob', eval_metric='mlogloss',
        n_jobs=-1, random_state=42, verbosity=0)),
]
if HAS_CB:
    model_defs.append(('cb', lambda: CatBoostClassifier(
        iterations=300, learning_rate=0.05, depth=6,
        random_state=42, verbose=0, thread_count=-1)))
model_defs.extend([
    ('lr', lambda: LogisticRegression(
        C=1.0, max_iter=1000, solver='lbfgs',
        random_state=42, n_jobs=-1)),
    ('rf', lambda: RandomForestClassifier(
        n_estimators=300, max_depth=15, n_jobs=-1, random_state=42)),
])

oof_dict = {}
test_dict = {}
per_model_f1 = {}

skf = StratifiedKFold(5, shuffle=True, random_state=42)
for name, ctor in model_defs:
    print(f'\n--- Model {name} ---')
    t0 = time.time()
    oof = np.zeros((N, K))
    test_p = np.zeros((len(Xte_i), K))
    fold_f1 = []
    for fold, (tr, va) in enumerate(skf.split(Xtr_i, y)):
        m = ctor()
        if name.startswith('xgb'):
            m.fit(Xtr_i[tr], y[tr] - MIN_CLS)
        else:
            m.fit(Xtr_i[tr], y[tr])
        p_va = m.predict_proba(Xtr_i[va])
        p_te = m.predict_proba(Xte_i)
        if name.startswith('xgb'):
            oof[va] = p_va
            test_p += p_te / 5
        else:
            col_idx = [int(np.where(m.classes_ == c)[0][0]) for c in CLASSES]
            oof[va] = p_va[:, col_idx]
            test_p += p_te[:, col_idx] / 5
        yp = CLASSES[np.argmax(oof[va], axis=1)]
        fold_f1.append(f1_score(y[va], yp, average='macro', zero_division=0))

    oof_dict[name] = oof
    test_dict[name] = test_p
    per_model_f1[name] = float(np.mean(fold_f1))
    print(f'  CV F1 = {per_model_f1[name]:.4f}  ({time.time()-t0:.0f}s)')

print('\nPer-model CV F1:')
for n, f in sorted(per_model_f1.items(), key=lambda kv: -kv[1]):
    print(f'  {n:<8} {f:.4f}')

In [ ]:
# Stacking via LR meta-learner on concatenated OOF probabilities
meta_X = np.hstack([oof_dict[n] for n in oof_dict]).astype(np.float32)
meta_Xte = np.hstack([test_dict[n] for n in oof_dict]).astype(np.float32)

stacked_oof = np.zeros((N, K))
stacked_test = np.zeros((len(Xte_i), K))

for tr, va in skf.split(meta_X, y):
    meta = LogisticRegression(C=1.0, max_iter=1000, random_state=42, solver='lbfgs')
    meta.fit(meta_X[tr], y[tr])
    col_idx = [int(np.where(meta.classes_ == c)[0][0]) for c in CLASSES]
    stacked_oof[va] = meta.predict_proba(meta_X[va])[:, col_idx]
    stacked_test += meta.predict_proba(meta_Xte)[:, col_idx] / 5

yp_stacked = CLASSES[np.argmax(stacked_oof, axis=1)]
stacked_f1 = f1_score(y, yp_stacked, average='macro', zero_division=0)
stacked_kappa = cohen_kappa_score(y, yp_stacked, weights='quadratic')
print(f'Stacked blend CV F1:  {stacked_f1:.4f}')
print(f'Stacked blend κ:       {stacked_kappa:.4f}')

In [ ]:
from sklearn.metrics import confusion_matrix
import matplotlib.colors as mcolors
cm = confusion_matrix(y, yp_stacked, labels=[1,2,3,4,5])
cm_norm = cm / cm.sum(axis=1, keepdims=True)

vg_cmap = mcolors.LinearSegmentedColormap.from_list(
    'vg_rust', ['#F2EBD9', VANGOGH['rust']])

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(cm_norm, cmap=vg_cmap, vmin=0, vmax=1, aspect='equal')
ax.set_xticks(range(5), [f'ESI-{i}' for i in range(1,6)])
ax.set_yticks(range(5), [f'ESI-{i}' for i in range(1,6)])
ax.set_xlabel('Predicted'); ax.set_ylabel('True')
for i in range(5):
    for j in range(5):
        c = 'white' if cm_norm[i,j] > 0.4 else '#3A3530'
        ax.text(j, i, f'{cm_norm[i,j]:.2f}', ha='center', va='center',
                fontsize=9, color=c)
ax.set_title('Kaggle blend confusion matrix (row-normalised)',
             fontsize=11, color='#3A3530', pad=10)
plt.colorbar(im, ax=ax, label='Recall')
ax.set_facecolor('#F2EBD9'); fig.patch.set_facecolor('#F2EBD9')
plt.tight_layout(); plt.show()

In [ ]:
# Nelder-Mead per-class threshold optimisation
def neg_f1(log_mult):
    mult = np.exp(log_mult)
    yp = CLASSES[np.argmax(stacked_oof * mult, axis=1)]
    return -f1_score(y, yp, average='macro', zero_division=0)

best_f1, best_mult = -1.0, np.ones(K)
rng = np.random.RandomState(42)
for restart in range(30):
    x0 = np.zeros(K) if restart == 0 else rng.randn(K) * 0.3
    res = minimize(neg_f1, x0, method='Nelder-Mead',
                   options={'maxiter': 500, 'xatol': 1e-4, 'fatol': 1e-6})
    if -res.fun > best_f1:
        best_f1 = -res.fun
        best_mult = np.exp(res.x)

print(f'Pre-threshold F1:  {stacked_f1:.4f}')
print(f'Optimised F1:      {best_f1:.4f}')
print(f'Multipliers:       {best_mult.round(4).tolist()}')

# 6. MIMIC-IV-ED validation: real clinical data

To validate my methodology on real clinical data, I applied the same pipeline
to MIMIC-IV-ED (Xie et al. 2022): **418 100 ED visits from Beth Israel Deaconess
Medical Center**. Running that pipeline on Kaggle would exceed the 9-hour limit
and require PhysioNet data access, so I include pre-computed results here.

The MIMIC cohort has a dramatically different class distribution than the Kaggle
synthetic data:

> **Data disclosure.** MIMIC-IV-ED data were accessed under the PhysioNet Credentialed Data Use Agreement (v3.1). Raw patient data are not redistributed. All MIMIC results presented below are aggregate statistics and model performance metrics. Access: https://physionet.org/content/mimic-iv-ed/


In [ ]:
try:
    display(Image(SUPP_DATA / 'figures' / 'class_distribution_comparison.png'))
except FileNotFoundError:
    print('[figure unavailable — supplementary dataset not attached]')

In [ ]:
# Load MIMIC pre-computed results
mimic = json.load(open(SUPP_DATA / 'results' / 'pipeline_results.json'))

print(f"MIMIC-IV-ED Blend-v2 Results (5-fold CV, N = {mimic['n_rows']:,}):")
print(f"  Embeddings:                {mimic['embedding_type']}")
print(f"  Feature count:             {mimic['feature_count']}")
print(f"  Blend models:              {mimic['n_models_in_blend']}")
print(f"  Macro F1 (stacked):        {mimic['cv_macro_f1_mean']:.4f}")
print(f"  Macro F1 (threshold-opt):  {mimic['threshold_optimised_f1']:.4f}")
print(f"  Cohen's quadratic κ:       {mimic['cohen_kappa']:.4f}")
print(f"  AUROC (macro OVR):          {mimic['auroc_macro']:.4f}")
print(f"  Per-class F1: {mimic['per_class_f1']}")

In [ ]:
try:
    display(Image(SUPP_DATA / 'figures' / 'confusion_matrix_combined.png'))
except FileNotFoundError:
    print('[figure unavailable]')

## Literature benchmarking

My κ = 0.701 sits firmly within the published range of human inter-rater
reliability for ED triage:

| Benchmark | κ |
|---|---|
| Mirhaghi et al. 2015 meta-analysis (40 579 cases, 19 studies) | 0.791 |
| **Live-patient nurse-to-expert agreement** | **0.694** ← my model at 0.701 |
| Scenario-based expert agreement | 0.824 |
| Travers et al. 2009 pediatric live triage | 0.57 |

The MIMIC-IV-ED pipeline uses a 5-model blend (4 LightGBM variants +
1 internally stacked LGB+XGB+CB ensemble) with convex weight optimisation
and per-class multiplicative threshold shifts. Most published ML triage
systems (Levin et al. 2018; Raita et al. 2019; Kwon et al. 2018; Ivanov KATE)
predict **binary** outcomes: admission vs discharge, or critical vs
non-critical. 5-class ESI prediction at >400k scale is, to my knowledge,
a first. Binary-outcome comparisons are not directly applicable.

# 7. Clinical safety: asymmetric triage risk

## Motivation from the literature

- **Under-triage is associated with higher mortality**: OR 1.24–2.18 across
  published studies (Haas et al. 2010; Rubano et al. 2016).
- **The American College of Surgeons Committee on Trauma (ACS-COT)** codifies
  **≤5% under-triage as the clinical target**, explicitly tolerating 25–50%
  over-triage. This 10:1 asymmetry reflects that under-triage kills while
  over-triage merely consumes resources.
- **Sax et al. (2023)** (5.3 M ED encounters) found ESI sensitivity for
  high-acuity patients was only 65.9%, with documented subgroup disparities
  in under-triage risk.

Standard macro-F1 optimisation ignores this asymmetry. I tested four mechanisms.

In [ ]:
try:
    display(Image(SUPP_DATA / 'figures' / 'clinical_operating_curve.png'))
except FileNotFoundError:
    print('[figure unavailable]')

In [ ]:
try:
    display(Image(SUPP_DATA / 'figures' / 'ordinal_penalty_comparison.png'))
except FileNotFoundError:
    print('[figure unavailable]')

## 7.1 Three training-time variants

| Variant | F1 | κ | Under-triage % | Severe UT % |
|---|---|---|---|---|
| A: Standard (blend-v2 baseline) | 0.591 | 0.702 | 15.2% | 0.57% |
| B: Asymmetric class weights (5:3:1:1:1) | 0.547 | 0.659 | **7.7%** | 0.20% |
| **C: Directional sample weights (3× for under-triage)** | 0.475 | 0.634 | **6.8%** | **0.18%** |

Variant C **cuts under-triage by 55%** (15.2% → 6.8%) and **severe under-triage
by 68%** at an F1 cost of 0.116.

In [ ]:
try:
    display(Image(SUPP_DATA / 'figures' / 'ordinal_penalty_comparison.png'))
except FileNotFoundError:
    print('[figure unavailable]')

## 7.2 KEY FINDING: threshold optimisation and asymmetric training are substitutes, not complements

Applying Nelder-Mead threshold optimisation *after* directional training (Variant C)
**inverts the safety asymmetry**: C+thresh returns under-triage to 15.5%, chasing
max-F1 again. The two mechanisms are alternative ways to move along the same
F1-vs-UT Pareto frontier, not additive.

In [ ]:
# 20-configuration asymmetric-weight Pareto sweep
sweep = pd.read_csv(SUPP_DATA / 'results' / 'asymmetric_weight_sweep.csv')
pareto = sweep[sweep['is_pareto_optimal']].sort_values('undertriage_rate')
print('Pareto-optimal asymmetric weight configurations:')
print(pareto[['config_name', 'esi1_w', 'esi2_w', 'esi3_w',
              'macro_f1', 'undertriage_rate', 'severe_undertriage_rate']]
      .to_string(index=False))

In [ ]:
try:
    display(Image(SUPP_DATA / 'figures' / 'pareto_training_vs_threshold.png'))
except FileNotFoundError:
    print('[figure unavailable]')

## 7.3 Calibration and the ACS-COT target

Per-class isotonic calibration (nested 5-fold on OOF probabilities)
marginally improves threshold-optimised macro F1 but *hurts* raw argmax F1:

| Metric | Uncalibrated | Calibrated |
|---|---|---|
| F1 (raw argmax) | 0.591 | 0.577 |
| F1 (Nelder-Mead thresh) | 0.597 | **0.5985** |
| First mult with UT ≤ 5% | **3.0× (UT=4.87%, F1=0.459)** | 3.25× (UT=4.96%, F1=0.440) |

The uncalibrated operating curve actually reaches the ACS-COT 5% target
slightly earlier (mult=3.0, F1=0.459) than the calibrated one (mult=3.25,
F1=0.440). Calibration's value is confined to the threshold-optimised
operating point (+0.0016 F1) and to producing more honest probability
estimates for downstream uses. The mechanism: isotonic calibration corrects
systematic over-confidence in the high-acuity classes (ESI-1/2), which
removes "lucky" correct argmax classifications but enables slightly better
per-class threshold optimisation.

In [ ]:
try:
    display(Image(SUPP_DATA / 'figures' / 'operating_curve_calibrated_vs_original.png'))
except FileNotFoundError:
    print('[figure unavailable]')

# 8. Explainability & fairness

## Regulatory context

The EU AI Act (Regulation 2024/1689) classifies *"emergency healthcare patient
triage"* as **high-risk AI** in Annex III, Section 5(d). Compliance is required
by **August 2, 2026**. My system addresses four articles:

- **Article 9** (risk management): systematic bias auditing below
- **Article 10** (bias monitoring): intersectional disparity tracking
- **Article 13** (transparency): per-patient SHAP explanations
- **Article 14** (human oversight): operating curve (§7) lets clinicians choose
  the safety vs performance tradeoff

## Global SHAP analysis (MIMIC-IV-ED, 20 000 stratified subsample)

In [ ]:
try:
    display(Image(SUPP_DATA / 'figures' / 'shap_beeswarm.png'))
    display(Image(SUPP_DATA / 'figures' / 'shap_bar_global.png'))
except FileNotFoundError:
    print('[figures unavailable]')

Text-embedding features (`emb_pca_0`, `emb_pca_1`) dominate; chief complaint
carries the most individual signal. Clinical vitals (SBP, O2sat, temperature,
HR) occupy the next tier, consistent with how emergency physicians triage.

Non-clinical features (`language`, `insurance_type`) also contribute, raising
fairness concerns under EU AI Act Article 10(2)(f–g).

## Per-patient waterfall examples

In [ ]:
# Model working well on a routine case
try:
    display(Image(SUPP_DATA / 'figures' / 'shap_waterfall_esi3.png'))
except FileNotFoundError:
    print('[figure unavailable]')

*Left panel, ESI-3 confidently correct*: abnormal vitals push the prediction toward ESI-3; stable vitals and text embedding reinforce it. This is clinically intuitive.

*Right panel, ESI-3 misclassified*: despite concerning features, text embedding and partial vitals steered the prediction toward a lower acuity. This type of case is exactly where a clinician should override the model.


In [ ]:
try:
    display(Image(SUPP_DATA / 'figures' / 'shap_waterfall_esi2.png'))
except FileNotFoundError:
    print('[figure unavailable]')

*ESI-2 demographic-proxy concern*: the model over-weights non-clinical
features (insurance, language). This is a fairness red flag that only per-patient
SHAP can expose.

## Intersectional bias audit

In [ ]:
try:
    display(Image(SUPP_DATA / 'figures' / 'bias_audit_disparities.png'))
except FileNotFoundError:
    print('[figure unavailable]')

- Single-axis age disparity: 1.71× ratio between 18–30 and 80+ bands
- Sex disparity: 1.02× (negligible)
- Race disparity: 1.30×, preliminary (race mapping is incomplete)
- The asymmetric 2.5× threshold reduces overall UT from 15.2% to 6.1%, but
  the absolute reduction is smaller in the 18–30 band than in the 80+ band,
  so the relative age disparity widens even as absolute UT falls in every
  band. This is the fairness-safety tension reported in the safety chapter
  of the full report.

## Compensated shock: the mechanism behind the age bias

In [ ]:
try:
    display(Image(SUPP_DATA / 'figures' / 'age_undertriage_vitals_comparison.png'))
except FileNotFoundError:
    print('[figure unavailable]')

Among true ESI-1/2 patients (those who really need rapid resuscitation):

| Age group | N high-acuity | % with both HR AND SBP normal |
|---|---|---|
| 18–30 | ~21 k | **49.48%** |
| 31–65 | ~77 k | 42.3% |
| 66+ | ~65 k | **37.15%** |

**Half of critically ill young patients present with both vitals normal.**
This is the physiological phenomenon of *compensated shock*: young
cardiovascular systems maintain haemodynamics longer than elderly ones.
The model cannot flag ESI-1/2 from vitals alone for this cohort.

## Mitigation attempt: age-adjusted vital z-scores

In [ ]:
fe = json.load(open(SUPP_DATA / 'results' / 'feature_engineering_results.json'))
# Note: 'baseline_f1' in the JSON stores the blend F1 (0.5912), not the
# single-model baseline. The single-model baseline F1 is 0.5562.
single_model_baseline = 0.5562
print(f"Single-model baseline (65 features):  F1 = {single_model_baseline:.4f}")
print(f"Augmented single-model (77 features): F1 = {fe['augmented_f1']:.4f} "
      f"(Δ = {fe['augmented_f1'] - single_model_baseline:+.4f})")
if fe.get('augmented_asym_metrics'):
    a = fe['augmented_asym_metrics']
    print(f"Augmented + {a['weights_config']} weights: "
          f"F1 = {a['f1']:.4f}, UT = {a['undertriage_rate']*100:.2f}%")
if fe.get('augmented_age_group_ut'):
    print(f"Age-group UT (augmented): {fe['augmented_age_group_ut']}")
print(f"New features in top-20 importance: {fe['new_features_in_top20']}")

The 12 age-adjusted features (vital z-scores by age decile, REMS score,
comorbidity×vital interactions) add only **+0.0007 F1** to single-model
performance, within noise. However, two of them (`hr_age_zscore`, `n_age_abnormal_vitals`)
rank in the top 20 importance; they carry signal but compete with raw vitals.
Combined with Mild+ asymmetric weights, overall UT reaches **11.2%**
with flattened age-group disparities (18–30: 12.9%, 66+: 9.8%).

**Honest reading**: the compensated-shock mechanism is real (§8 figure above)
but the simplest engineering mitigations don't close the F1 gap. A future
direction is to combine augmented features with the full multi-model blend
(not tested here).

# 9. Submission generation

I generate two submissions following the strategy described in §2:

1. **`submission.csv`**: the threshold-optimised full blend (§5). This is the
   primary submission, leveraging both text and tabular features.
2. **`submission_honest_tabular.csv`**: a tabular-only baseline for comparison,
   reflecting what an honest clinical model without leakage would look like.

Both CSVs are saved to `/kaggle/working/` in the competition-required format
(`patient_id, triage_acuity`).

In [ ]:
# ── 9.1 Primary submission: threshold-optimised full blend ──────────────
test_preds = CLASSES[np.argmax(stacked_test * best_mult, axis=1)]

submission = pd.DataFrame({
    'patient_id': df_test['patient_id'].values,
    'triage_acuity': test_preds.astype(int),
})

# Align to sample_submission.csv ordering
sample_sub = pd.read_csv(COMP_DATA / 'sample_submission.csv')
submission = sample_sub[['patient_id']].merge(submission, on='patient_id', how='left')
if submission['triage_acuity'].isna().any():
    mode = int(pd.Series(test_preds).mode().iloc[0])
    submission['triage_acuity'] = submission['triage_acuity'].fillna(mode).astype(int)

submission.to_csv(OUTPUT / 'submission.csv', index=False)
print(f'submission.csv  shape={submission.shape}')
print('Class distribution:')
print(submission['triage_acuity'].value_counts().sort_index())

In [ ]:
# ── 9.2 Honest tabular baseline (no text embeddings) ──────────────────
from lightgbm import LGBMClassifier
# Note: the ~0.873 F1 claim for the honest tabular baseline comes from the
# 5-fold CV analysis of the tabular-only pipeline (see §2 leakage discovery).
# This cell trains on the full training set for submission purposes only —
# no CV is run here. See `results/kaggle/honest_tabular_results.json` for the
# full CV numbers.

lgbm_params = dict(n_estimators=200, learning_rate=0.05, num_leaves=63,
                    subsample=0.8, colsample_bytree=0.8, bagging_freq=1,
                    n_jobs=-1, random_state=42, verbose=-1)

tab_cols = Xtr_tab.columns.tolist()
Xtr_tab_i = imp.fit_transform(Xtr_tab.values)
Xte_tab_i = imp.transform(Xte_tab.values)

m = LGBMClassifier(**lgbm_params)
m.fit(Xtr_tab_i, y)
honest_preds = m.predict(Xte_tab_i)

honest_sub = pd.DataFrame({
    'patient_id': df_test['patient_id'].values,
    'triage_acuity': honest_preds.astype(int),
})
honest_sub = sample_sub[['patient_id']].merge(honest_sub, on='patient_id', how='left')
if honest_sub['triage_acuity'].isna().any():
    mode = int(pd.Series(honest_preds).mode().iloc[0])
    honest_sub['triage_acuity'] = honest_sub['triage_acuity'].fillna(mode).astype(int)

honest_sub.to_csv(OUTPUT / 'submission_honest_tabular.csv', index=False)
print(f'submission_honest_tabular.csv  shape={honest_sub.shape}')
print('Class distribution:')
print(honest_sub['triage_acuity'].value_counts().sort_index())

**Disposition note**: both CSVs are available in the notebook output. I
submit the full-pipeline CSV (`submission.csv`) for the leaderboard, but the
honest tabular baseline captures what a clinical system should actually
deploy: its ~0.87 macro F1 on this synthetic data reflects genuine clinical
prediction, not keyword recovery.

# 10. Limitations & future work

## Limitations

1. **ESI-5 remains difficult** (F1 ≈ 0.26 on MIMIC). With 0.3% prevalence in
   real data, I cannot meaningfully learn this class without active labelling.
2. **No variant reaches ACS-COT ≤5% under-triage without substantial F1 cost**.
   Calibration + thresholds gets to UT = 4.96%
   at F1 = 0.440. The 5% target may have been
   calibrated for binary trauma triage, not 5-class ESI.
3. **Asymmetric threshold widens the relative age disparity** even as
   absolute UT falls in every age band. The fairness-safety tension is
   discussed further in §8.
4. **Single-institution MIMIC data** (Beth Israel Deaconess). External validation
   on a second health system is essential before clinical deployment.
5. **Text embeddings dominate predictions**, limiting the clinical
   interpretability I'd expect from a decision-support system.
6. **Non-clinical features contribute** (language, insurance_type). This is a
   well-known failure mode for EHR-based ML; requires demographic-agnostic
   training or adversarial debiasing.

## Future work

- Age-adjusted vital features inside the full blend (not tested here)
- Custom LightGBM ordinal objective with gradient + hessian for native
  directional penalty
- External validation on a second institution's ED cohort
- CORAL/CORN ordinal deep networks (Cao et al. 2020)
- Real-time SHAP dashboard for per-patient explanations
- Tier-1 deterministic recall for exact-match complaint strings (a significant
  ~25% coverage opportunity in synthetic Kaggle data, less so in real data)

# 11. References

**Clinical evidence**
- Haas, B., et al. (2010). Survival of the fittest: the hidden cost of
  undertriage of major trauma. *J Am Coll Surg* 211(6): 804–811.
- Mirhaghi, A., et al. (2015). Reliability of the Emergency Severity Index:
  meta-analysis. *Sultan Qaboos Univ Med J* 15(1): e71–e77.
- Rubano, J. A., et al. (2016). Undertriage in the trauma system: a national
  analysis. *J Trauma Acute Care Surg* 81(6): 1142–1149.
- Sax, D. R., et al. (2023). Evaluation of Emergency Severity Index triage
  system accuracy in large healthcare system. *JAMA Netw Open* 6(3): e233404.
- Travers, D. A., et al. (2009). Reliability and validity assessment of a new
  five-level pediatric ESI. *Acad Emerg Med* 16(9): 843–849.
- Wuerz, R. C., et al. (2000). Reliability and validity of a new five-level
  triage instrument. *Acad Emerg Med* 7(3): 236–242.

**RETTS**
- Widgren, B. R., & Jourak, M. (2011). Medical Emergency Triage and Treatment
  System (METTS). *J Emerg Med* 40(6): 623–628.
- Wireklint, C., et al. (2022). Risk factors for 7-day mortality following
  triage with RETTS. *Scand J Trauma Resusc Emerg Med* 30: 12.

**Models and methods**
- Alsentzer, E., et al. (2019). Publicly available clinical BERT embeddings.
  *NAACL 2019 Clinical NLP Workshop*: 72–78.
- Cao, W., Mirjalili, V., & Raschka, S. (2020). Rank-consistent ordinal
  regression for neural networks. *Pattern Recogn Lett* 140: 325–331.
- Ivanov, O., et al. (2021). Improving ED triage acuity assignment with
  ML and NLP. *J Emerg Nurs* 47(2): 265–278.
- Kwon, J. M., et al. (2018). Deep-learning-based out-of-hospital cardiac
  arrest prognostic system. *Resuscitation* 139: 84–91.
- Levin, S., et al. (2018). Machine-learning-based electronic triage more
  accurately differentiates patients by clinical outcomes. *Ann Emerg Med* 71(5):
  565–574.
- Raita, Y., et al. (2019). ED triage prediction of clinical outcomes using
  ML. *Crit Care* 23: 64.
- Xie, F., et al. (2022). MIMIC-IV-ED: a freely accessible emergency
  department database. *Sci Data* 9: 658.

**Regulatory**
- Regulation (EU) 2024/1689 of the European Parliament and of the Council
  on Artificial Intelligence (EU AI Act), Annex III §5(d).
- American College of Surgeons Committee on Trauma (ACS-COT). *Resources
  for Optimal Care of the Injured Patient*, 6th ed.: under-triage ≤5% target.